# Final Experimental Setup Results Reader

Notebook para leer todas las tablas, comparar corridas y ver Monte Carlo por metodo y por corrida completa.

In [1]:
from pathlib import Path
import json

import pandas as pd
import plotly.express as px
from IPython.display import HTML, Markdown, display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "portafolios").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "portafolios").exists():
    raise RuntimeError("Could not locate the repository root containing the portafolios package.")

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "data_exports" / "final_experimental_setup"
TABLE_DIR = OUTPUT_DIR / "tables"
CURATED_RUNS_DIR = OUTPUT_DIR / "framework_runs"
LIVE_RUNS_DIR = PROJECT_ROOT / "outputs" / "final_experimental_setup" / "framework_runs"

def resolve_run_dir(run_id: str) -> Path:
    live = LIVE_RUNS_DIR / run_id
    curated = CURATED_RUNS_DIR / run_id
    if (live / "plots" / "constructions").exists():
        return live
    if curated.exists():
        return curated
    if live.exists():
        return live
    raise FileNotFoundError(f"Could not find a saved run directory for {run_id}.")


In [2]:

summary = pd.read_csv(TABLE_DIR / "final_experiment_summary.csv")
backtest = pd.read_csv(TABLE_DIR / "backtest_summary.csv")
mc = pd.read_csv(TABLE_DIR / "mc_summary.csv")
concentration = pd.read_csv(TABLE_DIR / "concentration.csv")
mc_terminal_values = pd.read_csv(TABLE_DIR / "mc_terminal_values.csv")

with (OUTPUT_DIR / "experiment_config.json").open(encoding="utf-8") as f:
    config = json.load(f)

print(f"Runs: {summary['run_id'].nunique()} | Method rows: {len(summary)}")
print(f"MC terminal rows: {len(mc_terminal_values)}")
print(f"Universes: {', '.join(sorted(summary['universe_id'].unique()))}")

Runs: 27 | Method rows: 135
MC terminal rows: 270000
Universes: djia, multi_asset_etf, sp100_style_top100


## Config

In [3]:
pd.Series({
    "source": config.get("source"),
    "construction_dates": config.get("construction_dates"),
    "estimation_windows_months": config.get("estimation_windows_months"),
    "evaluation": config.get("evaluation"),
    "mc_simulations": config.get("mc_simulations"),
    "mc_seed": config.get("mc_seed"),
    "completed_runs": len(config.get("completed_runs", [])),
}, name="value")

source                                                      yfinance
construction_dates              [2019-06-01, 2020-06-01, 2022-01-03]
estimation_windows_months                               [12, 24, 36]
evaluation                   12-month forward out-of-sample backtest
mc_simulations                                                  2000
mc_seed                                                           42
completed_runs                                                    27
Name: value, dtype: object

## Aggregate Table

In [4]:
summary.head(20)

,run_id,source,universe_id,universe_display_name,requested_tickers,available_tickers,n_assets,construction_date,estimation_months,construction_start,construction_end,backtest_start,backtest_end,mc_horizon,mc_simulations,mc_seed,methods,hca_n_clusters,method,total_return,annualized_return,volatility,sharpe_ratio,max_drawdown,mean_terminal_value,median_terminal_value,std_terminal_value,prob_loss,herfindahl,max_weight
0,djia_20190601_12m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,12,2018-06-01,2019-05-31,2019-06-03,2020-06-01,252,2000,42,"['equal_weight', 'naive_risk_parity', 'markowi...",5,equal_weight,0.1532,0.1532,0.3332,0.4598,-0.3285,1.1113,1.0988,0.1745,0.2725,0.0333,0.0333
1,djia_20190601_12m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,12,2018-06-01,2019-05-31,2019-06-03,2020-06-01,252,2000,42,"['equal_weight', 'naive_risk_parity', 'markowi...",5,naive_risk_parity,0.1218,0.1218,0.3271,0.3725,-0.3252,1.1376,1.1216,0.1689,0.2020,0.0351,0.0458
2,djia_20190601_12m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,12,2018-06-01,2019-05-31,2019-06-03,2020-06-01,252,2000,42,"['equal_weight', 'naive_risk_parity', 'markowi...",5,markowitz,0.0792,0.0792,0.2888,0.2744,-0.2741,1.3923,1.3829,0.1811,0.0115,0.2666,0.3801
3,djia_20190601_12m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,12,2018-06-01,2019-05-31,2019-06-03,2020-06-01,252,2000,42,"['equal_weight', 'naive_risk_parity', 'markowi...",5,hrp_recursive,0.1011,0.1011,0.2813,0.3596,-0.2803,1.2355,1.2257,0.1455,0.0415,0.0968,0.1484
4,djia_20190601_12m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,12,2018-06-01,2019-05-31,2019-06-03,2020-06-01,252,2000,42,"['equal_weight', 'naive_risk_parity', 'markowi...",5,hca_deprado_ew_nrp,0.1104,0.1104,0.2917,0.3786,-0.2751,1.2190,1.2091,0.1512,0.0630,0.0836,0.1782
5,djia_20190601_24m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,24,2017-06-01,2019-05-31,2019-06-03,2020-06-01,252,2000,42,"['equal_weight', 'naive_risk_parity', 'markowi...",5,equal_weight,0.1532,0.1532,0.3332,0.4598,-0.3285,1.1738,1.1585,0.1674,0.1470,0.0333,0.0333
6,djia_20190601_24m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,24,2017-06-01,2019-05-31,2019-06-03,2020-06-01,252,2000,42,"['equal_weight', 'naive_risk_parity', 'markowi...",5,naive_risk_parity,0.1250,0.1250,0.3283,0.3809,-0.3271,1.1684,1.1524,0.1638,0.1425,0.0347,0.0490
7,djia_20190601_24m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,24,2017-06-01,2019-05-31,2019-06-03,2020-06-01,252,2000,42,"['equal_weight', 'naive_risk_parity', 'markowi...",5,markowitz,0.0073,0.0073,0.3608,0.0203,-0.3756,1.2808,1.2677,0.1893,0.0495,0.1368,0.2173
8,djia_20190601_24m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,24,2017-06-01,2019-05-31,2019-06-03,2020-06-01,252,2000,42,"['equal_weight', 'naive_risk_parity', 'markowi...",5,hrp_recursive,0.0945,0.0945,0.2992,0.3160,-0.3037,1.1552,1.1482,0.1361,0.1260,0.0890,0.2034
9,djia_20190601_24m,yfinance,djia,DJIA Constituents,"['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...","['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', '...",30,2019-06-01,24,2017-06-01,2019-05-31,2019-06-

## Winners By Run

In [5]:
def winner_rows(df, metric, best="max"):
    idx = df.groupby("run_id")[metric].idxmax() if best == "max" else df.groupby("run_id")[metric].idxmin()
    out = df.loc[idx, ["run_id", "universe_id", "construction_date", "estimation_months", "method", metric]].copy()
    return out.rename(columns={"method": f"best_{metric}_method", metric: f"best_{metric}"})

winners = winner_rows(summary, "sharpe_ratio", "max")
for metric, best in [
    ("annualized_return", "max"),
    ("volatility", "min"),
    ("max_drawdown", "max"),
    ("prob_loss", "min"),
    ("herfindahl", "min"),
]:
    winners = winners.merge(
        winner_rows(summary, metric, best).drop(columns=["universe_id", "construction_date", "estimation_months"]),
        on="run_id",
        how="left",
    )

winners.sort_values(["universe_id", "construction_date", "estimation_months"])

,run_id,universe_id,construction_date,estimation_months,best_sharpe_ratio_method,best_sharpe_ratio,best_annualized_return_method,best_annualized_return,best_volatility_method,best_volatility,best_max_drawdown_method,best_max_drawdown,best_prob_loss_method,best_prob_loss,best_herfindahl_method,best_herfindahl
0,djia_20190601_12m,djia,2019-06-01,12,equal_weight,0.4598,equal_weight,0.1532,hrp_recursive,0.2813,markowitz,-0.2741,markowitz,0.0115,equal_weight,0.0333
1,djia_20190601_24m,djia,2019-06-01,24,hca_deprado_ew_nrp,0.4781,equal_weight,0.1532,hca_deprado_ew_nrp,0.2916,hca_deprado_ew_nrp,-0.2884,markowitz,0.0495,equal_weight,0.0333
2,djia_20190601_36m,djia,2019-06-01,36,equal_weight,0.4598,equal_weight,0.1532,hrp_recursive,0.2894,hrp_recursive,-0.2857,markowitz,0.0285,equal_weight,0.0333
3,djia_20200601_12m,djia,2020-06-01,12,equal_weight,2.4967,markowitz,0.7870,hrp_recursive,0.1529,naive_risk_parity,-0.0819,markowitz,0.0385,equal_weight,0.0333
4,djia_20200601_24m,djia,2020-06-01,24,equal_weight,2.4967,equal_weight,0.4130,hrp_recursive,0.1347,naive_risk_parity,-0.0815,markowitz,0.1205,equal_weight,0.0333
5,djia_20200601_36m,djia,2020-06-01,36,equal_weight,2.4967,equal_weight,0.4130,hrp_recursive,0.1340,naive_risk_parity,-0.0817,markowitz,0.1110,equal_weight,0.0333
6,djia_20220103_12m,djia,2022-01-03,12,hca_deprado_ew_nrp,-0.2803,hca_deprado_ew_nrp,-0.0598,naive_risk_parity,0.1946,naive_risk_parity,-0.2015,markowitz,0.0000,equal_weight,0.0333
7,djia_20220103_24m,djia,2022-01-03,24,hca_deprado_ew_nrp,-0.3194,hca_deprado_ew_nrp,-0.0610,hca_deprado_ew_nrp,0.1909,hca_deprado_ew_nrp,-0.1797,markowitz,0.0615,equal_weight,0.0333
8,djia_20220103_36m,djia,2022-01-03,36,hca_deprado_ew_nrp,-0.3440,hca_deprado_ew_nrp,-0.0661,hrp_recursive,0.1889,hrp_recursive,-0.1806,markowitz,0.0380,equal_weight,0.0333
9,multi_asset_etf_20190601_12m,multi_asset_etf,2019-06-01,12,hca_deprado_ew_nrp,1.7399,hca_deprado_ew_nrp,0.1802,markowitz,0.0683,markowitz,-0.0865,markowitz,0.0035,equal_weight,0.0769


## Average Method Ranking

In [6]:
summary.groupby("method").agg(
    runs=("run_id", "count"),
    mean_annualized_return=("annualized_return", "mean"),
    mean_volatility=("volatility", "mean"),
    mean_sharpe=("sharpe_ratio", "mean"),
    mean_max_drawdown=("max_drawdown", "mean"),
    mean_prob_loss=("prob_loss", "mean"),
    mean_herfindahl=("herfindahl", "mean"),
).sort_values("mean_sharpe", ascending=False)

,runs,mean_annualized_return,mean_volatility,mean_sharpe,mean_max_drawdown,mean_prob_loss,mean_herfindahl
method,,,,,,,
equal_weight,27,0.1376,0.2073,0.8327,-0.1908,0.2082,0.0401
naive_risk_parity,27,0.1155,0.1878,0.7828,-0.1769,0.1780,0.0506
hca_deprado_ew_nrp,27,0.0991,0.1890,0.6170,-0.1836,0.1668,0.0839
hrp_recursive,27,0.0653,0.1910,0.5497,-0.1887,0.1430,0.1288
markowitz,27,0.0911,0.2389,0.4238,-0.2205,0.0348,0.3289


## Select One Run

In [7]:
RUN_ID = "djia_20190601_24m"
run_table = summary.loc[summary["run_id"] == RUN_ID].copy()
run_table[[
    "run_id",
    "universe_id",
    "construction_date",
    "estimation_months",
    "method",
    "n_assets",
    "hca_n_clusters",
    "annualized_return",
    "volatility",
    "sharpe_ratio",
    "max_drawdown",
    "prob_loss",
    "herfindahl",
    "max_weight",
]].sort_values("sharpe_ratio", ascending=False)

,run_id,universe_id,construction_date,estimation_months,method,n_assets,hca_n_clusters,annualized_return,volatility,sharpe_ratio,max_drawdown,prob_loss,herfindahl,max_weight
9,djia_20190601_24m,djia,2019-06-01,24,hca_deprado_ew_nrp,30,5,0.1394,0.2916,0.4781,-0.2884,0.1195,0.0639,0.1012
5,djia_20190601_24m,djia,2019-06-01,24,equal_weight,30,5,0.1532,0.3332,0.4598,-0.3285,0.1470,0.0333,0.0333
6,djia_20190601_24m,djia,2019-06-01,24,naive_risk_parity,30,5,0.1250,0.3283,0.3809,-0.3271,0.1425,0.0347,0.0490
8,djia_20190601_24m,djia,2019-06-01,24,hrp_recursive,30,5,0.0945,0.2992,0.3160,-0.3037,0.1260,0.0890,0.2034
7,djia_20190601_24m,djia,2019-06-01,24,markowitz,30,5,0.0073,0.3608,0.0203,-0.3756,0.0495,0.1368,0.2173


## Run-Level Performance Bars

In [8]:
metric = "sharpe_ratio"
fig = px.bar(
    run_table.sort_values(metric, ascending=False),
    x="method",
    y=metric,
    color="method",
    title=f"{RUN_ID}: {metric}",
)
fig.update_layout(showlegend=False)
fig.show()

## Monte Carlo Comparison For The Selected Run

Este es el equivalente comparativo de los histogramas del experimento anterior, pero ahora usando todas las simulaciones terminales del `run_id` elegido.

In [9]:
mc_run = mc_terminal_values.loc[mc_terminal_values["run_id"] == RUN_ID].copy()
fig = px.histogram(
    mc_run,
    x="terminal_value",
    color="method",
    histnorm="probability density",
    barmode="overlay",
    nbins=60,
    opacity=0.45,
    marginal="box",
    title=f"Monte Carlo terminal distribution comparison: {RUN_ID}",
)
fig.show()

## Monte Carlo Summary For The Selected Run

In [10]:
mc.loc[mc["run_id"] == RUN_ID].sort_values("prob_loss")

,run_id,universe_id,construction_date,estimation_months,method,mean_terminal_value,median_terminal_value,std_terminal_value,prob_loss
7,djia_20190601_24m,djia,2019-06-01,24,markowitz,1.2808,1.2677,0.1893,0.0495
9,djia_20190601_24m,djia,2019-06-01,24,hca_deprado_ew_nrp,1.1602,1.1552,0.1376,0.1195
8,djia_20190601_24m,djia,2019-06-01,24,hrp_recursive,1.1552,1.1482,0.1361,0.1260
6,djia_20190601_24m,djia,2019-06-01,24,naive_risk_parity,1.1684,1.1524,0.1638,0.1425
5,djia_20190601_24m,djia,2019-06-01,24,equal_weight,1.1738,1.1585,0.1674,0.1470


## Compare A Metric Across All Runs

In [11]:
METRIC = "max_drawdown" 
fig = px.line(
    summary.sort_values(["universe_id", "estimation_months", "construction_date", "method"]),
    x="construction_date",
    y=METRIC,
    color="method",
    facet_row="universe_id",
    facet_col="estimation_months",
    markers=True,
    title=f"{METRIC} by universe, estimation window, and construction date",
)
fig.update_layout(height=900)
fig.show()

## Inline Saved Plots For The Selected Run

This section embeds the saved HTML plots for the selected `RUN_ID`. When the live thesis run folder has the richer cleaned plot tree, the notebook uses that folder automatically; otherwise it falls back to the curated export copy.


In [12]:
def first_existing(*paths: Path) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None

def comparison_plot_paths(run_id: str) -> list[tuple[str, Path]]:
    run_dir = resolve_run_dir(run_id)
    specs = [
        ("Backtest comparison", first_existing(run_dir / "plots" / "comparisons" / "backtest_comparison.html", run_dir / "plots" / "backtest_comparison.html")),
        ("MC terminal comparison", first_existing(run_dir / "plots" / "comparisons" / "mc_terminal_comparison.html", run_dir / "plots" / "mc_terminal_comparison.html")),
    ]
    return [(label, path) for label, path in specs if path is not None]

def method_plot_paths(run_id: str, method: str) -> list[tuple[str, Path]]:
    run_dir = resolve_run_dir(run_id)
    specs = [
        ("Weights bar", first_existing(run_dir / "plots" / "constructions" / method / "weights_bar.html", run_dir / "constructions" / method / "weights_bar.html")),
        ("Weights pie", first_existing(run_dir / "plots" / "constructions" / method / "weights_pie.html", run_dir / "constructions" / method / "weights_pie.html")),
        ("Weights scatter", first_existing(run_dir / "plots" / "constructions" / method / "weights_scatter.html", run_dir / "constructions" / method / "weights_scatter.html")),
        ("Weights bubble", first_existing(run_dir / "plots" / "constructions" / method / "weights_bubble.html")),
        ("Backtest", first_existing(run_dir / "plots" / "backtests" / method / "backtest.html", run_dir / "backtests" / method / "backtest.html")),
        ("Drawdown", first_existing(run_dir / "plots" / "backtests" / method / "drawdown.html")),
        ("MC distribution", first_existing(run_dir / "plots" / "monte_carlo" / method / "mc_distribution.html", run_dir / "monte_carlo" / method / "mc_distribution.html")),
        ("MC paths", first_existing(run_dir / "plots" / "monte_carlo" / method / "mc_paths.html", run_dir / "monte_carlo" / method / "mc_paths.html")),
        ("HRP dendrogram", first_existing(run_dir / "plots" / "constructions" / method / "hrp_dendrogram.html")),
        ("HRP distance matrix", first_existing(run_dir / "plots" / "constructions" / method / "hrp_distance_matrix.html")),
        ("HRP distance histogram", first_existing(run_dir / "plots" / "constructions" / method / "hrp_distance_histogram.html")),
    ]
    if "markowitz" in method.lower():
        specs.insert(4, ("Efficient frontier", first_existing(run_dir / "plots" / "constructions" / method / "efficient_frontier.html")))
    return [(label, path) for label, path in specs if path is not None]

def iframe_height(label: str) -> int:
    if label in {"MC paths", "Backtest", "Drawdown", "Efficient frontier", "Backtest comparison", "MC terminal comparison"}:
        return 720
    if "HRP" in label:
        return 760
    return 640

def iframe_for_plot(path: Path, label: str) -> str:
    height = iframe_height(label)
    return (
        f"<details open style='margin: 0.75rem 0 1.5rem 0;'>"
        f"<summary style='font-weight: 600; margin-bottom: 0.5rem;'>{label}</summary>"
        f"<iframe src='{path.resolve().as_uri()}' width='100%' height='{height}' "
        "style='border: 1px solid #d0d7de; border-radius: 8px; background: white;'></iframe>"
        "</details>"
    )

run_dir = resolve_run_dir(RUN_ID)
display(pd.Series({
    "run_id": RUN_ID,
    "plot_source_dir": str(run_dir.relative_to(PROJECT_ROOT)),
    "methods": sorted(summary.loc[summary["run_id"] == RUN_ID, "method"].unique()),
}, name="plot_source"))

comparison_html = "".join(iframe_for_plot(path, label) for label, path in comparison_plot_paths(RUN_ID))
if comparison_html:
    display(Markdown("### Comparison Plots"))
    display(HTML(comparison_html))

for method in sorted(summary.loc[summary["run_id"] == RUN_ID, "method"].unique()):
    plots = method_plot_paths(RUN_ID, method)
    if not plots:
        continue
    display(Markdown(f"### {method}"))
    display(HTML("".join(iframe_for_plot(path, label) for label, path in plots)))


run_id                                             djia_20190601_24m
plot_source_dir    outputs\final_experimental_setup\framework_run...
methods            [equal_weight, hca_deprado_ew_nrp, hrp_recursi...
Name: plot_source, dtype: object

### Comparison Plots

### equal_weight

### hca_deprado_ew_nrp

### hrp_recursive

### markowitz

### naive_risk_parity

## Links To Saved HTML Plots

In [13]:
def html_link(path: Path, label: str | None = None) -> str:
    label = label or path.name
    return f'<a href="{path.resolve().as_uri()}" target="_blank">{label}</a>'

def links_for_run(run_id: str) -> HTML:
    run_dir = resolve_run_dir(run_id)
    rows = [f"<li><b>Plot source directory</b>: {run_dir.relative_to(PROJECT_ROOT)}</li>"]
    for label, path in comparison_plot_paths(run_id):
        rows.append(f"<li>{html_link(path, label)}</li>")

    comparison_csv = first_existing(
        run_dir / "monte_carlo" / "comparison_terminal_values.csv",
        run_dir / "plots" / "mc_terminal_values.csv",
    )
    if comparison_csv is not None:
        rows.append(f"<li>{html_link(comparison_csv, 'MC terminal values CSV')}</li>")

    for method in sorted(summary.loc[summary["run_id"] == run_id, "method"].unique()):
        rows.append(f"<li><b>{method}</b></li>")
        for label, path in method_plot_paths(run_id, method):
            rows.append(f"<li style='margin-left: 1.5rem'>{html_link(path, label)}</li>")
    return HTML("<ul>" + "".join(rows) + "</ul>")

links_for_run(RUN_ID)


## All Run IDs

In [14]:
pd.Series(sorted(summary["run_id"].unique()), name="run_id").to_frame()

,run_id
0,djia_20190601_12m
1,djia_20190601_24m
2,djia_20190601_36m
3,djia_20200601_12m
4,djia_20200601_24m
5,djia_20200601_36m
6,djia_20220103_12m
7,djia_20220103_24m
8,djia_20220103_36m
9,multi_asset_etf_20190601_12m
